# 02 · EDA, Feature Engineering & SMOTE

This notebook does three things:

1. **EDA** -- four diagnostic plots that characterise the class imbalance,
   amount distributions, inter-feature correlations, and temporal fraud patterns
2. **Feature engineering** -- six new features derived from `Time` and `Amount`
3. **SMOTE** -- synthetic oversampling applied to the *training set only*

> **Why SMOTE only on the training fold?**
> SMOTE interpolates between real fraud transactions and their k-nearest
> neighbours.  If applied before splitting, synthetic training samples would be
> blended from test-set fraud points -- the model would then be evaluated on
> derivatives of its own training data, producing inflated metrics that don't
> reflect production performance.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from imblearn.over_sampling import SMOTE
from pathlib import Path

plt.style.use("seaborn-v0_8-whitegrid")
DATA_DIR  = Path("../data")
PLOTS_DIR = Path("../plots")
PLOTS_DIR.mkdir(exist_ok=True)
COLORS = ["#2196F3", "#F44336"]   # blue = legit, red = fraud


In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")

fraud = train[train["Class"] == 1]
legit = train[train["Class"] == 0]

print(f"Train: {len(train):,} rows  |  fraud: {len(fraud)}  legit: {len(legit)}")
print(f"Test : {len(test):,} rows   |  fraud: {test['Class'].sum()}  legit: {(test['Class']==0).sum()}")


Train: 227,845 rows  |  fraud: 394  legit: 227451
Test : 56,962 rows   |  fraud: 98  legit: 56864


## Plot 1 · Class Imbalance Bar Chart

The most important chart to show stakeholders first -- it immediately explains
why standard accuracy is useless and why every engineering decision that follows
is made with the minority class in mind.


In [3]:
fig, ax = plt.subplots(figsize=(7, 5))
counts = [len(legit), len(fraud)]
labels = ["Legit (0)", "Fraud (1)"]
bars = ax.bar(labels, counts, color=COLORS, width=0.4, edgecolor="white")
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1500,
            f"{count:,}\n({count/len(train)*100:.2f}%)",
            ha="center", va="bottom", fontweight="bold")
ax.set_title("Class Imbalance", fontsize=15, fontweight="bold")
ax.set_ylabel("Transaction Count")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_ylim(0, max(counts) * 1.15)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "1_class_imbalance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Insight: Fraud is 0.17% of transactions (394 in train). A naive 'always legit'")
print("classifier scores 99.83% accuracy but catches zero fraud -- accuracy is meaningless.")


Insight: Fraud is 0.17% of transactions (394 in train). A naive 'always legit'
classifier scores 99.83% accuracy but catches zero fraud -- accuracy is meaningless.


## Plot 2 · Amount Distribution by Class (KDE, log scale)

Fraud and legitimate transactions have very different amount profiles.
A log scale reveals structure across several orders of magnitude.


In [4]:
fig, ax = plt.subplots(figsize=(9, 5))
for data, label, color in [
    (legit["Amount"], "Legit", COLORS[0]),
    (fraud["Amount"], "Fraud", COLORS[1]),
]:
    sns.kdeplot(np.log1p(data), ax=ax, label=label,
                color=color, fill=True, alpha=0.35, linewidth=2)

xtick_vals = [0, 1, 5, 10, 50, 100, 500, 1000, 5000]
ax.set_xticks(np.log1p(xtick_vals))
ax.set_xticklabels([f"${v:,}" for v in xtick_vals], rotation=30, ha="right")
ax.set_title("Transaction Amount by Class (log1p scale)", fontweight="bold")
ax.set_xlabel("Amount"); ax.set_ylabel("Density"); ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "2_amount_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Insight: Fraud median=${fraud['Amount'].median():.2f} vs legit median=${legit['Amount'].median():.2f}.")
print("Fraud clusters at small amounts -- consistent with card-testing behaviour.")


Insight: Fraud median=$9.17 vs legit median=$22.00.
Fraud clusters at small amounts -- consistent with card-testing behaviour.


## Plot 3 · Correlation Heatmap (Fraud Rows Only)

Because V1-V28 are PCA-derived, most pairs should be near-orthogonal.
Correlations *within the fraud subset* reveal structure that doesn't appear
in the full dataset -- potential candidates for interaction features.


In [5]:
features = [f"V{i}" for i in range(1, 29)] + ["Amount"]
corr = fraud[features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, ax=ax,
            cbar_kws={"shrink": 0.8, "label": "Pearson r"})
ax.set_title("Feature Correlation -- Fraud Transactions Only",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "3_fraud_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

corr_vals = corr.where(~mask).stack()
top_pair  = corr_vals.abs().idxmax()
top_val   = corr_vals[top_pair]
print(f"Insight: Strongest pair in fraud rows: {top_pair[0]} <-> {top_pair[1]} (r={top_val:.2f}).")
print("Most V-features are near-independent by PCA design, but V17<->V18 leaked correlation")
print("within fraud -- a candidate for an interaction feature.")


Insight: Strongest pair in fraud rows: V18 <-> V17 (r=0.97).
Most V-features are near-independent by PCA design, but V17<->V18 leaked correlation
within fraud -- a candidate for an interaction feature.


## Plot 4 · Fraud Rate by Hour of Day

`Time` is seconds elapsed since the first transaction -- not a wall-clock timestamp.
But modulo 24 hours it still captures intra-day fraud patterns.


In [6]:
train["hour"] = (train["Time"] // 3600 % 24).astype(int)
hourly = train.groupby("hour")["Class"].agg(["sum", "count"])
hourly["fraud_rate"] = hourly["sum"] / hourly["count"] * 100

peak_hour = hourly["fraud_rate"].idxmax()
peak_rate = hourly["fraud_rate"].max()
low_hour  = hourly["fraud_rate"].idxmin()
low_rate  = hourly["fraud_rate"].min()

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(hourly.index, hourly["fraud_rate"], color="#E91E63", edgecolor="white")
ax.bar(peak_hour, peak_rate, color="#880E4F", edgecolor="white",
       label=f"Peak: {peak_hour}:00 ({peak_rate:.2f}%)")
ax.set_title("Fraud Rate by Hour of Day", fontsize=14, fontweight="bold")
ax.set_xlabel("Hour (UTC)"); ax.set_ylabel("Fraud Rate (%)")
ax.set_xticks(range(24))
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f%%"))
ax.legend(); plt.tight_layout()
plt.savefig(PLOTS_DIR / "4_fraud_rate_by_hour.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Insight: Fraud peaks at {peak_hour}:00 UTC ({peak_rate:.2f}%) vs lowest at "
      f"{low_hour}:00 UTC ({low_rate:.2f}%).")
print("A 34x swing -- hour_of_day will be a strong engineered feature.")

train.drop(columns=["hour"], inplace=True)  # clean up temp column


Insight: Fraud peaks at 2:00 UTC (1.80%) vs lowest at 10:00 UTC (0.05%).
A 34x swing -- hour_of_day will be a strong engineered feature.


## Feature Engineering

Six new features are derived from `Time` and `Amount`.  They are created for
*both* splits using the same function so there is no transform mismatch at inference.

| Feature | Derivation | Rationale |
|---|---|---|
| `hour_of_day` | `Time // 3600 % 24` | Captures the intra-day fraud cycle seen above |
| `is_night` | `hour_of_day in [0-5]` | Binary flag for the high-risk overnight window |
| `log_amount` | `log(Amount + 1)` | Compresses the heavy right tail of Amount |
| `is_round_amount` | `Amount % 1 == 0` | Round-dollar amounts correlate with card testing |
| `amount_rolling_mean` | 5-row rolling mean | Local context for Amount magnitude |
| `amount_rolling_std` | 5-row rolling std | Local volatility in Amount |


In [7]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["hour_of_day"]         = (out["Time"] // 3600 % 24).astype(int)
    out["is_night"]            = out["hour_of_day"].between(0, 5).astype("int8")
    out["log_amount"]          = np.log1p(out["Amount"])
    out["is_round_amount"]     = (out["Amount"] % 1 == 0).astype("int8")
    out["amount_rolling_mean"] = out["Amount"].rolling(5, min_periods=1).mean()
    out["amount_rolling_std"]  = out["Amount"].rolling(5, min_periods=1).std().fillna(0)
    return out

train_eng = engineer_features(train)
test_eng  = engineer_features(test)

new_cols = ["hour_of_day","is_night","log_amount",
            "is_round_amount","amount_rolling_mean","amount_rolling_std"]
print("New columns added:")
for col in new_cols:
    print(f"  {col:<24}  sample: {train_eng[col].head(3).tolist()}")

train_eng.to_csv(DATA_DIR / "train_engineered.csv", index=False)
test_eng.to_csv(DATA_DIR / "test_engineered.csv",   index=False)
print(f"\nSaved train_engineered.csv  {train_eng.shape}")
print(f"Saved test_engineered.csv   {test_eng.shape}")


New columns added:
  hour_of_day               sample: [20, 10, 11]
  is_night                  sample: [0, 0, 0]
  log_amount                sample: [2.1186622548331173, 1.3837912309017721, 5.17105201550216]
  is_round_amount           sample: [0, 0, 0]
  amount_rolling_mean       sample: [7.32, 5.155, 61.803333333333335]
  amount_rolling_std        sample: [0.0, 3.061772362537751, 98.14167429452857]



Saved train_engineered.csv  (227845, 37)
Saved test_engineered.csv   (56962, 37)


## SMOTE Resampling

SMOTE generates synthetic minority-class samples by interpolating between real
fraud points and their k-nearest neighbours.

**Applied to the training set only** -- the test set stays untouched at its
natural 0.17% fraud rate, which is what the model will see in production.

If SMOTE were applied before splitting, synthetic training samples would contain
information from test-set fraud transactions, making evaluation metrics optimistically
biased and unreliable as a proxy for real-world performance.


In [8]:
FEAT_COLS = [f"V{i}" for i in range(1, 29)] + [
    "hour_of_day","is_night","log_amount",
    "is_round_amount","amount_rolling_mean","amount_rolling_std",
]
TARGET = "Class"
DROP   = ["Time", "Amount"]

X_train = train_eng.drop(columns=[TARGET] + DROP)
y_train = train_eng[TARGET]
X_test  = test_eng.drop(columns=[TARGET] + DROP)
y_test  = test_eng[TARGET]

print("Before SMOTE:")
print(f"  Train fraud  : {y_train.sum():>7,}  ({y_train.mean()*100:.4f}%)")
print(f"  Train legit  : {(y_train==0).sum():>7,}")
print(f"  Train total  : {len(y_train):>7,}")


Before SMOTE:
  Train fraud  :     394  (0.1729%)
  Train legit  : 227,451
  Train total  : 227,845


In [9]:
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)

print("After SMOTE (training set only):")
print(f"  Train fraud  : {(y_res==1).sum():>7,}  ({(y_res==1).mean()*100:.4f}%)")
print(f"  Train legit  : {(y_res==0).sum():>7,}  ({(y_res==0).mean()*100:.4f}%)")
print(f"  Train total  : {len(y_res):>7,}")
print()
print("Test set (unchanged):")
print(f"  Test fraud   : {y_test.sum():>7,}  ({y_test.mean()*100:.4f}%)")
print(f"  Test total   : {len(y_test):>7,}")
print()
print(f"Synthetic fraud rows created: {(y_res==1).sum() - y_train.sum():,}")


After SMOTE (training set only):
  Train fraud  : 227,451  (50.0000%)
  Train legit  : 227,451  (50.0000%)
  Train total  : 454,902

Test set (unchanged):
  Test fraud   :      98  (0.1720%)
  Test total   :  56,962

Synthetic fraud rows created: 227,057


In [10]:
X_res_df = pd.DataFrame(X_res, columns=X_train.columns)
X_res_df.to_csv(DATA_DIR / "X_train_resampled.csv", index=False)
pd.Series(y_res, name=TARGET).to_csv(DATA_DIR / "y_train_resampled.csv", index=False)
print("Saved X_train_resampled.csv and y_train_resampled.csv")


Saved X_train_resampled.csv and y_train_resampled.csv


## Summary

| Stage | Rows | Fraud | Fraud % |
|---|---|---|---|
| Raw dataset | 284,807 | 492 | 0.17% |
| Train (pre-SMOTE) | 227,845 | 394 | 0.17% |
| Train (post-SMOTE) | 454,902 | 227,451 | 50.00% |
| Test (never touched) | 56,962 | 98 | 0.17% |

**Next**: train and compare 6 classifiers on the SMOTE-resampled data.
